In [2]:
from datasets import load_dataset
import pandas as pd
from collections import Counter

f:\University\20252\Thesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
financebench = load_dataset("PatronusAI/financebench")
print(financebench)
print(financebench["train"].features)
print(financebench["train"][0])

DatasetDict({
    train: Dataset({
        features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
        num_rows: 150
    })
})
{'financebench_id': Value('string'), 'company': Value('string'), 'doc_name': Value('string'), 'question_type': Value('string'), 'question_reasoning': Value('string'), 'domain_question_num': Value('string'), 'question': Value('string'), 'answer': Value('string'), 'justification': Value('string'), 'dataset_subset_label': Value('string'), 'evidence': List({'evidence_text': Value('string'), 'doc_name': Value('string'), 'evidence_page_num': Value('int64'), 'evidence_text_full_page': Value('string')}), 'gics_sector': Value('string'), 'doc_type': Value('string'), 'doc_period': Value('int64'), 'doc_link': Value('string')}
{'financebench_id': 'financebench_id_03029', 'company': '3M

In [4]:
fb_df = financebench["train"].to_pandas()
print(fb_df.head(3))
print(fb_df.columns.tolist())
print(fb_df.shape)

         financebench_id company     doc_name      question_type  \
0  financebench_id_03029      3M  3M_2018_10K  metrics-generated   
1  financebench_id_04672      3M  3M_2018_10K  metrics-generated   
2  financebench_id_00499      3M  3M_2022_10K    domain-relevant   

                                 question_reasoning domain_question_num  \
0                            Information extraction                 NaN   
1                            Information extraction                 NaN   
2  Logical reasoning (based on numerical reasoning)                dg06   

                                            question  \
0  What is the FY2018 capital expenditure amount ...   
1  Assume that you are a public equities analyst....   
2  Is 3M a capital-intensive business based on FY...   

                                              answer  \
0                                           $1577.00   
1                                              $8.70   
2  No, the company is managing it

In [5]:
# Missing values
print(fb_df.isna().sum().sort_values(ascending=False))

# Basic length stats
fb_df["question_len"] = fb_df["question"].astype(str).str.len()
fb_df["answer_len"] = fb_df["answer"].astype(str).str.len()
fb_df["justification_len"] = fb_df["justification"].astype(str).str.len()

print(fb_df[["question_len", "answer_len", "justification_len"]].describe())

# Question type distribution
print(fb_df["question_type"].value_counts(dropna=False))

# Doc types / sectors
print(fb_df["doc_type"].value_counts(dropna=False))
print(fb_df["gics_sector"].value_counts(dropna=False))

domain_question_num     100
question_reasoning       50
justification            50
doc_name                  0
company                   0
question_type             0
financebench_id           0
question                  0
answer                    0
dataset_subset_label      0
evidence                  0
gics_sector               0
doc_type                  0
doc_period                0
doc_link                  0
dtype: int64
       question_len  answer_len  justification_len
count    150.000000  150.000000         100.000000
mean     161.093333   78.186667         225.680000
std       96.894902   96.555745         164.276982
min       44.000000    1.000000           9.000000
25%       80.000000    6.250000         100.750000
50%      137.500000   50.500000         163.500000
75%      210.750000  108.750000         391.750000
max      592.000000  609.000000         703.000000
question_type
metrics-generated    50
domain-relevant      50
novel-generated      50
Name: count, dtype: in

In [6]:
sample = fb_df.loc[0, ["question", "answer", "justification", "evidence", "doc_link"]]
print(sample["question"])
print(sample["answer"])
print(sample["justification"])
print(sample["doc_link"])
print(sample["evidence"])

What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
$1577.00
The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).
https://investors.3m.com/financials/sec-filings/content/0001558370-19-000470/0001558370-19-000470.pdf
[{'evidence_text': 'Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$\n5,058 \nAdjustments to reconcile net income including noncontrolling interest to net cash\nprovided by operating activities\n \n \n \n \n \n \n \nDepreciation and amortization\n \n \n1,488 \n \n1,544 \n \n1,474 \nCompany pension and postreti

In [7]:
fiqa_main = load_dataset("explodinggradients/fiqa", "main")
print(fiqa_main)
print(fiqa_main["train"].features)
print(fiqa_main["train"][0])

DatasetDict({
    train: Dataset({
        features: ['question', 'ground_truths'],
        num_rows: 5500
    })
    validation: Dataset({
        features: ['question', 'ground_truths'],
        num_rows: 500
    })
    test: Dataset({
        features: ['question', 'ground_truths'],
        num_rows: 648
    })
})
{'question': Value('string'), 'ground_truths': List(Value('string'))}
{'question': 'What is considered a business expense on a business trip?', 'ground_truths': ['The IRS Guidance pertaining to the subject.  In general the best I can say is your business expense may be deductible.  But it depends on the circumstances and what it is you want to deduct. Travel Taxpayers who travel away from home on business may deduct related   expenses, including the cost of reaching their destination, the cost   of lodging and meals and other ordinary and necessary expenses.   Taxpayers are considered “traveling away from home” if their duties   require them to be away from home substantia

In [8]:
fiqa_train = fiqa_main["train"].to_pandas()
fiqa_val = fiqa_main["validation"].to_pandas()
fiqa_test = fiqa_main["test"].to_pandas()

print(fiqa_train.shape, fiqa_val.shape, fiqa_test.shape)
print(fiqa_train.head(3))
print(fiqa_train.columns.tolist())

print(fiqa_train.isna().sum().sort_values(ascending=False))
print(fiqa_train["question"].astype(str).str.len().describe())
print(fiqa_train["ground_truths"].astype(str).str.len().describe())

(5500, 2) (500, 2) (648, 2)
                                            question  \
0  What is considered a business expense on a bus...   
1  Business Expense - Car Insurance Deductible Fo...   
2                     Starting a new online business   

                                       ground_truths  
0  [The IRS Guidance pertaining to the subject.  ...  
1  [As a general rule, you must choose between a ...  
2  [Most US states have rules that go something l...  
['question', 'ground_truths']
question         0
ground_truths    0
dtype: int64
count    5500.000000
mean       61.497636
std        22.772693
min        14.000000
25%        45.000000
50%        59.000000
75%        76.000000
max       158.000000
Name: question, dtype: float64
count     5500.000000
mean      2663.416182
std       2879.052793
min          4.000000
25%        908.750000
50%       1808.000000
75%       3362.000000
max      41639.000000
Name: ground_truths, dtype: float64


In [9]:
print(fiqa_train.isna().sum().sort_values(ascending=False))
print(fiqa_train["question"].astype(str).str.len().describe())
print(fiqa_train["ground_truths"].astype(str).str.len().describe())

corpus_df = fiqa_corpus["corpus"].to_pandas()
print(corpus_df.shape)
print(corpus_df.head(3))
print(corpus_df.isna().sum().sort_values(ascending=False))
print(corpus_df["doc"].astype(str).str.len().describe())

question         0
ground_truths    0
dtype: int64
count    5500.000000
mean       61.497636
std        22.772693
min        14.000000
25%        45.000000
50%        59.000000
75%        76.000000
max       158.000000
Name: question, dtype: float64
count     5500.000000
mean      2663.416182
std       2879.052793
min          4.000000
25%        908.750000
50%       1808.000000
75%       3362.000000
max      41639.000000
Name: ground_truths, dtype: float64


NameError: name 'fiqa_corpus' is not defined

In [ ]:
fiqa_ragas = load_dataset("explodinggradients/fiqa", "ragas_eval_v3")
print(fiqa_ragas)
print(fiqa_ragas["baseline"].features)
print(fiqa_ragas["baseline"][0])